# Why do BreakHis-trained models fail on BACH, and can label-free fixes recover it?
### Pre-registered cross-site experiment (companion to the main paper notebook)

**Before running (Kaggle):** GPU T4 · Internet ON · **Add Input**: `ambarish/breakhis` and
`erenyeager00111/iciar2018-bach-images` · then **Save Version → Save & Run All (Commit)**. About 3 h on a T4.
(Optional: `QUICK_TEST = True` first, ~30 min.)

## Pre-registration (written before any result of this notebook was seen)
**Observation from the main run:** trained on BreakHis, every model fell to 50–66 % balanced accuracy on BACH,
with high sensitivity and very low specificity, and AUC (≈ 0.75) far above balanced accuracy (≈ 0.62).

**Hypothesised causes and the label-free fix for each:**

| # | Cause | Fix (uses **no** BACH label) |
|---|---|---|
| H1 | **Scale mismatch.** BACH images (2048×1536, 0.42 µm/px; Aresta et al. 2019) were squeezed to 224 px, so tissue looks ~3× smaller than in BreakHis | **Scale-matched tiling.** Measure the median nucleus size in BreakHis 200X and in BACH from the haematoxylin channel, cut BACH into tiles covering the same physical field as a BreakHis 200X image, and average predictions over tiles |
| H2 | **Stain / scanner shift** | **Macenko stain normalisation** of BACH to a reference stain matrix estimated from BreakHis training images |
| H3 | **Label-prior shift.** 68 % of BreakHis images are malignant, BACH is 50 % | **EM prior-shift correction** (Saerens, Latinne & Decaestecker, *Neural Computation* 14(1):21–41, 2002) |
| H4 | **Label-space mismatch.** BreakHis has no normal tissue and no in-situ carcinoma; BACH has both | No fix. It is *measured*: the malignant-prediction rate per original BACH class, and a pre-specified label-compatible subset (Benign vs Invasive) |

**Design:** 4 inputs {resize (previous), tiles, stain-norm, tiles + stain-norm} × 2 post-hoc {none, EM} = 8
conditions, applied identically to **every** method (ours = LoRA Phikon-v2 + stain aug, 3 seeds; frozen
comparators). All models are trained on all of BreakHis (patient-level inner validation for early stopping and
temperature). BACH labels are used **only** to score.

**Primary endpoint:** BACH binary balanced accuracy. **Primary comparison:** ours with the full pipeline
(tiles + stain-norm + EM) vs ours with the previous pipeline (resize, none). The primary comparison is a
win only if the paired bootstrap 95 % CI is above 0 in **all 3 seeds**. Every condition is reported,
whatever the outcome.

## 1. Configuration

In [1]:
import os, sys, re, json, time, math, random, hashlib, subprocess, importlib, warnings, shutil, copy, pickle, itertools
from pathlib import Path
from collections import defaultdict

QUICK_TEST = False or os.environ.get('CMFE_QUICK', '0') == '1'
DRY_RUN = os.environ.get('CMFE_DRY_RUN', '0') == '1'      # internal code test only — leave as is
BREAKHIS_SLUG = 'ambarish/breakhis'
BACH_SLUG = 'erenyeager00111/iciar2018-bach-images'


def _ensure(pkgs):
    for pip_name, mod in pkgs:
        try:
            importlib.import_module(mod)
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

_ensure([('timm', 'timm'), ('transformers', 'transformers'), ('scikit-image', 'skimage')] + ([] if DRY_RUN else [('kagglehub', 'kagglehub')]))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import timm
from skimage.color import rgb2hed, hed2rgb
from skimage.filters import threshold_otsu
from skimage.measure import label as cc_label, regionprops
from skimage.morphology import binary_opening, disk
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, roc_curve
warnings.filterwarnings('ignore')

WORK = Path(os.environ.get('CMFE_WORK', '/kaggle/working'))
INPUT = Path(os.environ.get('CMFE_INPUT', '/kaggle/input'))
VAR_DIR = Path(os.environ.get('CMFE_VARDIR', '/tmp/bach_variants'))        # large temporary files, not saved as output
FEAT_DIR, RES_DIR, FIG_DIR, TAB_DIR = WORK / 'features', WORK / 'results', WORK / 'figures', WORK / 'tables'
for d in (FEAT_DIR, RES_DIR, FIG_DIR, TAB_DIR, VAR_DIR):
    d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    backbones=['dinov2_vitb14', 'phikon_v2'], batch_extract=32, num_workers=4,
    lora_backbone='phikon_v2', lora_r=8, lora_alpha=16, lora_dropout=0.1,
    lora_epochs=6, lora_patience=3, lora_bs=16, lora_lr=1e-4, head_lr=1e-3, lora_wd=1e-4, label_smooth=0.1,
    seeds=[0, 1, 2],
    hid=256, emb=128, epochs=80, patience=15, lr=1e-3, wd=1e-4, batch=256, p_in_drop=0.2,
    logreg_C=[0.01, 0.1, 1.0], svm_C=10.0, svm_pca=256, knn_k=10,
    ref_mag=200, n_scale_images=40, n_stain_ref=40, max_tiles=16, n_boot=2000,
)
if QUICK_TEST and not DRY_RUN:
    CFG.update(lora_epochs=1, seeds=[0], epochs=20, patience=5, n_boot=200, n_scale_images=10, n_stain_ref=10)
if DRY_RUN:
    CFG.update(lora_epochs=2, lora_patience=2, lora_bs=8, seeds=[0, 1], epochs=30, patience=10, n_boot=50, batch_extract=8,
               num_workers=0, svm_pca=16, batch=64, logreg_C=[0.1], n_scale_images=6, n_stain_ref=6)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(42)
print(f'QUICK_TEST={QUICK_TEST} | DRY_RUN={DRY_RUN} | device={DEVICE}')
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
json.dump(CFG, open(RES_DIR / 'config_bach.json', 'w'), indent=2)

QUICK_TEST=False | DRY_RUN=False | device=cuda
GPU: Tesla T4


## 2. Data: BreakHis (training) and BACH (test only)

In [2]:
SUBTYPES = ['A', 'F', 'PT', 'TA', 'DC', 'LC', 'MC', 'PC']
MAGS = [40, 100, 200, 400]
RX = re.compile(r'SOB_(?P<cls>[BM])_(?P<sub>[A-Z]+)-(?P<year>\d+)-(?P<pid>[0-9A-Za-z]+)-(?P<mag>\d+)-(?P<seq>\d+)\.png$', re.I)
EXT_LABEL = {'normal': 0, 'benign': 0, 'insitu': 1, 'in situ': 1, 'invasive': 1}


def _synth(rng, si, nuc_r, h, w):
    """Synthetic H&E-like image with pink stroma and purple 'nuclei' of radius nuc_r (code testing only)."""
    img = np.ones((h, w, 3), np.float32) * np.array([235, 190, 215]) + rng.normal(0, 8, (h, w, 3))
    yy, xx = np.mgrid[0:h, 0:w]
    for _ in range(int(h * w / (nuc_r ** 2 * 40)) + 5):
        cy, cx = rng.integers(0, h), rng.integers(0, w)
        m = (yy - cy) ** 2 + (xx - cx) ** 2 <= nuc_r ** 2
        img[m] = np.array([90 + 10 * si, 60, 140]) + rng.normal(0, 8, 3)
    return Image.fromarray(np.clip(img, 0, 255).astype(np.uint8))


def make_synthetic(root):
    root = Path(root)
    if root.exists():
        shutil.rmtree(root)
    rng, pid = np.random.default_rng(0), 1000
    for si, sub in enumerate(SUBTYPES):
        for _ in range(2):
            pid += 1
            for mag in MAGS:
                d = root / 'breast' / sub / f'p{pid}' / f'{mag}X'; d.mkdir(parents=True, exist_ok=True)
                for k in range(1, 4):
                    _synth(rng, si, 2 + mag // 40, 460, 700).save(d / f"SOB_{'B' if si < 4 else 'M'}_{sub}-14-{pid}-{mag}-{k:03d}.png")
    broot = root.parent / 'synthetic_bach'
    if broot.exists():
        shutil.rmtree(broot)
    for cls, si in [('Normal', 1), ('Benign', 2), ('InSitu', 5), ('Invasive', 6)]:
        d = broot / 'Photos' / cls; d.mkdir(parents=True, exist_ok=True)
        for k in range(6):
            _synth(rng, si, 3, 768, 1024).save(d / f'{cls[:2].lower()}{k:03d}.tif')   # nuclei ~0.6x of BreakHis 200X
    return root, broot


def find_attached(keyword):
    if not INPUT.exists():
        return []
    return [d for pat in ('*', '*/*', '*/*/*', '*/*/*/*') for d in INPUT.glob(pat) if d.is_dir() and keyword in d.name.lower()]


if DRY_RUN:
    BH_ROOT, BACH_ROOT = make_synthetic(WORK / 'synthetic_breakhis')
else:
    import kagglehub
    BH_ROOT = Path(kagglehub.dataset_download(BREAKHIS_SLUG))
    roots = find_attached('bach')
    BACH_ROOT = roots[0] if roots else Path(kagglehub.dataset_download(BACH_SLUG))
print('BreakHis:', BH_ROOT, '| BACH:', BACH_ROOT)

rows, seen = [], set()
for p in sorted(BH_ROOT.rglob('*.png')):
    m = RX.search(p.name)
    if not m or p.name in seen:
        continue
    seen.add(p.name); g = m.groupdict()
    rows.append(dict(path=str(p), file=p.name, subtype=g['sub'].upper(), y=int(g['cls'].upper() == 'M'), patient=f"{g['year']}-{g['pid']}", mag=int(g['mag'])))
df = pd.DataFrame(rows)
df = df[df.subtype.isin(SUBTYPES) & df.mag.isin(MAGS)].reset_index(drop=True)
if QUICK_TEST:
    df = df.groupby(['patient', 'mag']).head(4).reset_index(drop=True)


def pixel_hash(p):
    with Image.open(p) as im:
        return hashlib.md5(np.asarray(im.convert('RGB')).tobytes()).hexdigest()

df['hash'] = [pixel_hash(p) for p in df.path]          # same integrity cleanup as the main notebook
drop = set()
for h, g in df.groupby('hash'):
    if len(g) > 1:
        drop |= set(g.index) if g.subtype.nunique() > 1 else set(g.index[1:])
df = df.drop(index=sorted(drop)).reset_index(drop=True)
df['strat'] = df.patient.map(df.groupby('patient').subtype.agg(lambda s: SUBTYPES.index(s.mode()[0])))
print(f'BreakHis after cleanup: {len(df)} images, {df.patient.nunique()} patients ({len(drop)} duplicates removed)')

er = []
for p in sorted(BACH_ROOT.rglob('*')):
    if p.suffix.lower() not in ('.tif', '.tiff', '.png', '.jpg', '.jpeg') or RX.search(p.name):
        continue
    par = [a.name for a in p.parents if a.name.lower() in EXT_LABEL]
    if par:
        er.append(dict(path=str(p), cls={'in situ': 'InSitu'}.get(par[0].lower(), par[0]), y=EXT_LABEL[par[0].lower()]))
BACH = pd.DataFrame(er).drop_duplicates('path').reset_index(drop=True)
BACH['cls'] = BACH.cls.str.lower().map({'normal': 'Normal', 'benign': 'Benign', 'insitu': 'InSitu', 'invasive': 'Invasive'})
if QUICK_TEST:
    BACH = BACH.groupby('cls').head(20).reset_index(drop=True)
assert len(BACH) and BACH.y.nunique() == 2, 'BACH not found — attach it with Add Input'
print('BACH:', len(BACH), dict(BACH.cls.value_counts()))

# training / inner-validation split of ALL BreakHis (patient level), identical rule to the main notebook
PATIENT, PATHS, Y = df.patient.values, df.path.values, df.y.values
_all = np.arange(len(df))
for k in (5, 4, 3, 2):
    try:
        a, b = next(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=0).split(_all, df.strat.values, groups=PATIENT)); break
    except ValueError:
        continue
TR, VA = _all[a], _all[b]
PRIOR_TR = np.bincount(Y[TR], minlength=2) / len(TR)
print(f'train {len(TR)} | val {len(VA)} | training prior (benign, malignant) = {PRIOR_TR.round(3)}')

BreakHis: /kaggle/input/datasets/ambarish/breakhis | BACH: /kaggle/input/datasets/erenyeager00111/iciar2018-bach-images
BreakHis after cleanup: 7661 images, 80 patients (248 duplicates removed)
BACH: 400 {'Benign': np.int64(100), 'InSitu': np.int64(100), 'Invasive': np.int64(100), 'Normal': np.int64(100)}
train 5915 | val 1746 | training prior (benign, malignant) = [0.378 0.622]


## 3. H1 — scale estimation from nucleus size (label-free)
Haematoxylin channel → Otsu threshold → connected components → median equivalent diameter.
**Self-check:** BreakHis nucleus size should grow roughly with magnification (40X → 400X). If it does not,
the estimate is flagged and the scale factor falls back to 1.0 (tiles = BreakHis image size in BACH pixels).

In [3]:
def nucleus_diameter(path, max_side=1024):
    img = Image.open(path).convert('RGB')
    f = 1.0
    if max(img.size) > max_side:                          # speed: measure on a downscaled copy, then rescale
        f = max(img.size) / max_side
        img = img.resize((int(img.size[0] / f), int(img.size[1] / f)), Image.BILINEAR)
    h = rgb2hed(np.asarray(img).astype(np.float32) / 255.0)[..., 0]
    mask = binary_opening(h > threshold_otsu(h), disk(1))
    areas = np.array([r.area for r in regionprops(cc_label(mask))])
    areas = areas[(areas >= 6) & (areas <= mask.size * 0.02)]
    return np.nan if len(areas) < 5 else float(np.sqrt(np.median(areas) * 4 / np.pi) * f)


rng = np.random.default_rng(0)
scale_rows = []
for mag in MAGS:
    for p in rng.choice(df[df.mag == mag].path.values, min(CFG['n_scale_images'], (df.mag == mag).sum()), replace=False):
        scale_rows.append(dict(source=f'BreakHis {mag}X', mag=mag, d=nucleus_diameter(p)))
for p in rng.choice(BACH.path.values, min(CFG['n_scale_images'], len(BACH)), replace=False):
    scale_rows.append(dict(source='BACH', mag=np.nan, d=nucleus_diameter(p)))
SC = pd.DataFrame(scale_rows)
med = SC.groupby('source').d.median()
print('median nucleus diameter (px):\n', med.round(2).to_string())
bh = np.array([med.get(f'BreakHis {m}X', np.nan) for m in MAGS])
monotone = np.all(np.diff(bh) > 0)
corr = np.corrcoef(np.log(MAGS), np.log(bh))[0, 1] if np.all(np.isfinite(bh)) else np.nan
SCALE_OK = bool(monotone and corr > 0.9 and np.isfinite(med.get('BACH', np.nan)))
SCALE_R = float(med['BACH'] / med[f"BreakHis {CFG['ref_mag']}X"]) if SCALE_OK else 1.0
print(f'self-check: monotone with magnification={monotone}, log-log corr={corr:.3f} -> estimate valid: {SCALE_OK}')
print(f"scale factor r = BACH / BreakHis {CFG['ref_mag']}X = {SCALE_R:.3f}" + ('' if SCALE_OK else '  (FALLBACK — estimate failed the self-check)'))
TILE_W, TILE_H = int(round(700 * SCALE_R)), int(round(460 * SCALE_R))
print(f'BACH tile = {TILE_W}x{TILE_H} px (then resized to 700x460, the BreakHis image size)')
json.dump(dict(median_diam=med.to_dict(), monotone=bool(monotone), loglog_corr=float(corr), valid=SCALE_OK, r=SCALE_R,
               tile=[TILE_W, TILE_H]), open(RES_DIR / 'scale_estimate.json', 'w'), indent=2)

median nucleus diameter (px):
 source
BACH             14.97
BreakHis 100X     7.69
BreakHis 200X     9.29
BreakHis 400X     9.27
BreakHis 40X      5.29
self-check: monotone with magnification=False, log-log corr=0.936 -> estimate valid: False
scale factor r = BACH / BreakHis 200X = 1.000  (FALLBACK — estimate failed the self-check)
BACH tile = 700x460 px (then resized to 700x460, the BreakHis image size)


## 4. H2 — Macenko stain normalisation (reference estimated from BreakHis training images only)

In [4]:
def macenko_params(I, Io=240, alpha=1, beta=0.15):
    OD = -np.log((I.reshape(-1, 3).astype(np.float64) + 1) / Io)
    ODhat = OD[~np.any(OD < beta, axis=1)]
    if len(ODhat) < 100:
        raise ValueError('too few tissue pixels')
    _, eigvecs = np.linalg.eigh(np.cov(ODhat.T))
    That = ODhat @ eigvecs[:, 1:3]
    phi = np.arctan2(That[:, 1], That[:, 0])
    mn, mx = np.percentile(phi, alpha), np.percentile(phi, 100 - alpha)
    v1 = eigvecs[:, 1:3] @ np.array([np.cos(mn), np.sin(mn)])
    v2 = eigvecs[:, 1:3] @ np.array([np.cos(mx), np.sin(mx)])
    HE = np.array([v1, v2]).T if v1[0] > v2[0] else np.array([v2, v1]).T
    HE = HE * np.sign(HE.sum(0, keepdims=True) + 1e-12)
    C = np.linalg.lstsq(HE, OD.T, rcond=None)[0]
    return HE, np.percentile(C, 99, axis=1)


def macenko_normalize(img, HEref, maxCref, Io=240):
    I = np.asarray(img.convert('RGB'))
    try:
        HE, maxC = macenko_params(I)
    except Exception:
        return img                                        # not enough tissue: leave unchanged
    OD = -np.log((I.reshape(-1, 3).astype(np.float64) + 1) / Io)
    C = np.linalg.lstsq(HE, OD.T, rcond=None)[0] * (maxCref / (maxC + 1e-8))[:, None]
    out = np.clip(Io * np.exp(-HEref @ C), 0, 255).T.reshape(I.shape).astype(np.uint8)
    return Image.fromarray(out)


ref = []
for p in rng.choice(PATHS[TR], min(CFG['n_stain_ref'], len(TR)), replace=False):
    try:
        ref.append(macenko_params(np.asarray(Image.open(p).convert('RGB'))))
    except Exception:
        pass
HE_REF = np.median(np.stack([r[0] for r in ref]), 0)
MAXC_REF = np.median(np.stack([r[1] for r in ref]), 0)
print(f'stain reference from {len(ref)} BreakHis training images:\nHE=\n{HE_REF.round(3)}\nmaxC={MAXC_REF.round(3)}')

stain reference from 40 BreakHis training images:
HE=
[[0.561 0.316]
 [0.697 0.873]
 [0.431 0.358]]
maxC=[0.836 0.813]


## 5. Build the four BACH input variants (temporary JPEG q=95 files in /tmp, not saved as output)

In [5]:
def tiles_of(img):
    W, H = img.size
    tw, th = min(TILE_W, W), min(TILE_H, H)
    nx, ny = max(1, math.ceil(W / tw)), max(1, math.ceil(H / th))
    while nx * ny > CFG['max_tiles']:
        if nx >= ny: nx -= 1
        else: ny -= 1
    xs = np.linspace(0, W - tw, nx).astype(int) if nx > 1 else [int((W - tw) / 2)]
    ys = np.linspace(0, H - th, ny).astype(int) if ny > 1 else [int((H - th) / 2)]
    return [img.crop((x, y, x + tw, y + th)).resize((700, 460), Image.BICUBIC) for y in ys for x in xs]


VARIANTS = ['resize (previous)', 'tiles', 'stain-norm', 'tiles + stain-norm']
VAR_ITEMS = {v: [] for v in VARIANTS}        # list of (bach_index, path)
t0 = time.time()
for i, p in enumerate(BACH.path.values):
    img = Image.open(p).convert('RGB')
    img_n = macenko_normalize(img, HE_REF, MAXC_REF)
    VAR_ITEMS['resize (previous)'].append((i, p))
    fn = VAR_DIR / f'sn_{i:04d}.jpg'; img_n.save(fn, quality=95, subsampling=0); VAR_ITEMS['stain-norm'].append((i, str(fn)))
    for k, t in enumerate(tiles_of(img)):
        fn = VAR_DIR / f't_{i:04d}_{k:02d}.jpg'; t.save(fn, quality=95, subsampling=0); VAR_ITEMS['tiles'].append((i, str(fn)))
    for k, t in enumerate(tiles_of(img_n)):
        fn = VAR_DIR / f'tsn_{i:04d}_{k:02d}.jpg'; t.save(fn, quality=95, subsampling=0); VAR_ITEMS['tiles + stain-norm'].append((i, str(fn)))
    if i % 50 == 0:
        print(f'  variants: {i+1}/{len(BACH)}  {time.time()-t0:.0f}s')
print({v: len(it) for v, it in VAR_ITEMS.items()}, f'items | {time.time()-t0:.0f}s')

# qualitative figure: original vs stain-normalised vs one tile
fig, axes = plt.subplots(2, 4, figsize=(12, 5.2))
for r_, cls in enumerate(['Benign', 'Invasive']):
    i = int(BACH.index[BACH.cls == cls][0])
    img = Image.open(BACH.path[i]).convert('RGB')
    shows = [(img, 'BACH original'), (Image.open(VAR_ITEMS['stain-norm'][i][1]), 'Macenko-normalised'),
             (Image.open([p for j, p in VAR_ITEMS['tiles'] if j == i][0]), f'scale-matched tile (r={SCALE_R:.2f})'),
             (Image.open(df[df.mag == CFG['ref_mag']].path.values[0]).convert('RGB'), f"BreakHis {CFG['ref_mag']}X (reference)")]
    for c_, (im, ttl) in enumerate(shows):
        axes[r_, c_].imshow(im); axes[r_, c_].set_title(f'{ttl}' + (f'\n[{cls}]' if c_ < 3 else ''), fontsize=8); axes[r_, c_].axis('off')
fig.tight_layout(); fig.savefig(FIG_DIR / 'FB1_input_variants.png', dpi=300); fig.savefig(FIG_DIR / 'FB1_input_variants.pdf'); plt.close(fig)

  variants: 1/400  2s
  variants: 51/400  106s
  variants: 101/400  214s
  variants: 151/400  319s
  variants: 201/400  420s
  variants: 251/400  519s
  variants: 301/400  615s
  variants: 351/400  715s
{'resize (previous)': 400, 'tiles': 4800, 'stain-norm': 400, 'tiles + stain-norm': 4800} items | 811s


## 6. Models (same definitions as the main notebook)

In [6]:
IMNET = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))


def load_backbone(name):
    if name == 'dinov2_vitb14':
        m = timm.create_model('vit_small_patch14_dinov2' if DRY_RUN else 'vit_base_patch14_dinov2.lvd142m', pretrained=not DRY_RUN, num_classes=0, img_size=224)
        return (lambda x: m(x)), IMNET[0], IMNET[1], m
    if name == 'phikon_v2':
        from transformers import AutoModel, AutoImageProcessor, Dinov2Config, Dinov2Model
        if DRY_RUN:
            m = Dinov2Model(Dinov2Config(hidden_size=64, num_hidden_layers=2, num_attention_heads=2, intermediate_size=128, patch_size=16, image_size=224))
            mean, std = IMNET
        else:
            try:
                m = AutoModel.from_pretrained('owkin/phikon-v2', attn_implementation='sdpa')
            except Exception:
                m = AutoModel.from_pretrained('owkin/phikon-v2')
            proc = AutoImageProcessor.from_pretrained('owkin/phikon-v2'); mean, std = tuple(proc.image_mean), tuple(proc.image_std)
        return (lambda x: m(pixel_values=x).last_hidden_state[:, 0]), mean, std, m
    raise ValueError(name)


def hed_jitter(img, rng, sigma=0.05):
    a = np.asarray(img).astype(np.float32) / 255.0
    hed = rgb2hed(a)
    hed_j = hed * rng.uniform(1 - sigma, 1 + sigma, 3) + rng.uniform(-sigma, sigma, 3) * np.abs(hed).max(axis=(0, 1), keepdims=True)
    return Image.fromarray((np.clip(a + hed2rgb(hed_j) - hed2rgb(hed), 0, 1) * 255).astype(np.uint8))


def five_views(img):
    g = TF.to_tensor(img.resize((224, 224), Image.BICUBIC)); big = TF.to_tensor(img.resize((448, 448), Image.BICUBIC))
    return torch.stack([g] + [big[:, r:r + 224, c:c + 224] for r in (0, 224) for c in (0, 224)])


class ViewDS(Dataset):
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return five_views(Image.open(self.paths[i]).convert('RGB'))


class TrainDS(Dataset):
    def __init__(self, paths, y):
        self.paths, self.y = list(paths), np.asarray(y)
        self.rrc = T.RandomResizedCrop(224, scale=(0.25, 1.0), ratio=(0.75, 1.33))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        rng = np.random.default_rng()
        img = self.rrc(Image.open(self.paths[i]).convert('RGB')).rotate(90 * int(rng.integers(0, 4)))
        if rng.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        img = TF.adjust_contrast(TF.adjust_brightness(hed_jitter(img, rng), float(rng.uniform(0.9, 1.1))), float(rng.uniform(0.9, 1.1)))
        return TF.to_tensor(img), int(self.y[i])


@torch.no_grad()
def extract_frozen(name, paths, tag):
    cache = FEAT_DIR / f'{name}_{tag}.npz'
    if cache.exists():
        d = np.load(cache, allow_pickle=True)
        if list(d['paths']) == list(paths):
            return d['X']
    fn, mean, std, model = load_backbone(name)
    model.eval().to(DEVICE)
    mean, std = torch.tensor(mean, device=DEVICE).view(1, 3, 1, 1), torch.tensor(std, device=DEVICE).view(1, 3, 1, 1)
    feats = []
    for x in DataLoader(ViewDS(paths), batch_size=CFG['batch_extract'], num_workers=CFG['num_workers'], pin_memory=AMP):
        B = x.shape[0]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
            f = fn((x.view(B * 5, 3, 224, 224).to(DEVICE) - mean) / std).float().view(B, 5, -1)
        feats.append(torch.cat([f[:, 0], f[:, 1:].mean(1)], 1).cpu().numpy())
    X = np.concatenate(feats).astype(np.float32)
    np.savez(cache, X=X, paths=np.array(list(paths)))
    del model
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    return X


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        for p in base.parameters():
            p.requires_grad = False
        self.A, self.B = nn.Linear(base.in_features, r, bias=False), nn.Linear(r, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5)); nn.init.zeros_(self.B.weight)
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + self.B(self.A(self.drop(x))) * self.scale


def add_lora(model, r, alpha, dropout, targets=('query', 'key', 'value', 'q_proj', 'k_proj', 'v_proj', 'qkv')):
    n = 0
    for module in list(model.modules()):
        for cname, child in list(module.named_children()):
            if isinstance(child, nn.Linear) and cname in targets:
                setattr(module, cname, LoRALinear(child, r, alpha, dropout)); n += 1
    return n


class PeftNet(nn.Module):
    def __init__(self, backbone, n_cls):
        super().__init__()
        fn, mean, std, m = load_backbone(backbone)
        for p in m.parameters():
            p.requires_grad = False
        self.n_lora = add_lora(m, CFG['lora_r'], CFG['lora_alpha'], CFG['lora_dropout'])
        assert self.n_lora > 0
        self.m, self.fn = m, fn
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1)); self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))
        with torch.no_grad():
            D = fn(torch.zeros(1, 3, 224, 224)).shape[-1]
        self.head = nn.Sequential(nn.LayerNorm(D), nn.Dropout(0.1), nn.Linear(D, n_cls))

    def forward(self, x):
        return self.head(self.fn((x - self.mean) / self.std))


@torch.no_grad()
def peft_logits(model, paths):
    model.eval(); L = []
    for x in DataLoader(ViewDS(paths), batch_size=max(1, CFG['lora_bs'] // 2), num_workers=CFG['num_workers'], pin_memory=AMP):
        B = x.shape[0]
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
            L.append(model(x.view(B * 5, 3, 224, 224).to(DEVICE)).float().view(B, 5, -1).mean(1).cpu())
    return torch.cat(L).numpy()


def train_peft(seed):
    set_seed(seed)
    model = PeftNet(CFG['lora_backbone'], 2).to(DEVICE)
    lora_p = [p for n, p in model.named_parameters() if p.requires_grad and ('.A.' in n or '.B.' in n)]
    head_p = [p for n, p in model.named_parameters() if p.requires_grad and not ('.A.' in n or '.B.' in n)]
    opt = torch.optim.AdamW([{'params': lora_p, 'lr': CFG['lora_lr']}, {'params': head_p, 'lr': CFG['head_lr']}], weight_decay=CFG['lora_wd'])
    dl = DataLoader(TrainDS(PATHS[TR], Y[TR]), batch_size=CFG['lora_bs'], shuffle=True, num_workers=CFG['num_workers'],
                    pin_memory=AMP, drop_last=True, persistent_workers=CFG['num_workers'] > 0)
    E = CFG['lora_epochs']; total = max(2, E * len(dl)); warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    ce = nn.CrossEntropyLoss(weight=torch.tensor(1.0 / (2 * PRIOR_TR), dtype=torch.float32, device=DEVICE), label_smoothing=CFG['label_smooth'])
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    best, best_state, bad = (-1, 1e9), None, 0
    for ep in range(E):
        model.train(); t0 = time.time()
        for x, yb in dl:
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP):
                loss = ce(model(x.to(DEVICE)).float(), yb.to(DEVICE))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        lv = peft_logits(model, PATHS[VA])
        sc = (balanced_accuracy_score(Y[VA], lv.argmax(1)), F.cross_entropy(torch.tensor(lv), torch.tensor(Y[VA])).item())
        print(f'    seed {seed} epoch {ep+1}/{E}: val bal_acc {sc[0]:.4f} nll {sc[1]:.4f} ({time.time()-t0:.0f}s)')
        if sc[0] > best[0] + 1e-6 or (abs(sc[0] - best[0]) <= 1e-6 and sc[1] < best[1]):
            best, best_state, bad = sc, {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}, 0
        else:
            bad += 1
            if bad >= CFG['lora_patience']:
                break
    model.load_state_dict(best_state, strict=False)
    return model


class MLPHead(nn.Module):
    def __init__(self, D):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(CFG['p_in_drop']), nn.Linear(D, CFG['hid']), nn.LayerNorm(CFG['hid']), nn.GELU(), nn.Linear(CFG['hid'], CFG['emb']), nn.GELU(), nn.Linear(CFG['emb'], 2))

    def forward(self, x):
        return self.net(x)


def train_mlp(Xtr, ytr, Xva, yva, seed):
    set_seed(seed)
    model = MLPHead(Xtr.shape[1]).to(DEVICE)
    Xt, yt, Xv = torch.tensor(Xtr, device=DEVICE), torch.tensor(ytr, device=DEVICE), torch.tensor(Xva, device=DEVICE)
    ce = nn.CrossEntropyLoss(weight=torch.tensor(1.0 / (2 * PRIOR_TR), dtype=torch.float32, device=DEVICE), label_smoothing=CFG['label_smooth'])
    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    n, bs, best, best_state, bad = len(ytr), min(CFG['batch'], len(ytr)), -1, None, 0
    for ep in range(CFG['epochs']):
        model.train(); perm = torch.randperm(n, device=DEVICE)
        for s in range(max(1, n // bs)):
            b = perm[s * bs:(s + 1) * bs]; loss = ce(model(Xt[b]), yt[b]); opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            sc = balanced_accuracy_score(yva, model(Xv).argmax(1).cpu().numpy())
        if sc > best + 1e-6:
            best, best_state, bad = sc, copy.deepcopy(model.state_dict()), 0
        elif ep >= 10:
            bad += 1
            if bad >= CFG['patience']:
                break
    model.load_state_dict(best_state); model.eval()
    return lambda X: model(torch.tensor(X, device=DEVICE)).detach().cpu().numpy()

## 7. Train on all of BreakHis, predict every BACH variant

In [7]:
def softmax_np(l):
    e = np.exp(l - l.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def fit_temperature(lv, yv):
    Ts = np.exp(np.linspace(-2.5, 2.5, 101))
    return Ts[int(np.argmin([-np.log(softmax_np(lv / t)[np.arange(len(yv)), yv] + 1e-12).mean() for t in Ts]))]


def full_proba(clf, X):
    P = np.zeros((len(X), 2)); P[:, clf.classes_] = clf.predict_proba(X)
    return P


def aggregate(items, values, how):
    """Per-tile values -> per-image (mean of logits, or mean of probabilities)."""
    idx = np.array([i for i, _ in items])
    out = np.zeros((len(BACH), values.shape[1]))
    for i in range(len(BACH)):
        out[i] = values[idx == i].mean(0)
    return out


PRED = defaultdict(lambda: defaultdict(dict))      # PRED[method][variant][seed] -> calibrated BACH probabilities (N_bach x 2)
VAL_LOGITS = {}

# ---- frozen comparators
FE = {}
for name in CFG['backbones']:
    X = extract_frozen(name, PATHS, 'breakhis')
    sc = StandardScaler().fit(X[TR])
    FE[name] = dict(X=sc.transform(X).astype(np.float32), sc=sc,
                    V={v: sc.transform(extract_frozen(name, [p for _, p in VAR_ITEMS[v]], f'bach_{re.sub("[^a-z]", "", v)}')).astype(np.float32) for v in VARIANTS})
    print(f'[{name}] features ready')
Xc = np.concatenate([FE[n]['X'] for n in CFG['backbones']], 1)
Vc = {v: np.concatenate([FE[n]['V'][v] for n in CFG['backbones']], 1) for v in VARIANTS}


def fit_logreg(Xtr, ytr, Xva, yva):
    best = None
    for c in CFG['logreg_C']:
        clf = LogisticRegression(C=c, max_iter=3000, class_weight='balanced').fit(Xtr, ytr)
        s = balanced_accuracy_score(yva, clf.predict(Xva))
        if best is None or s > best[0]:
            best = (s, clf)
    return best[1]


def register(method, seed, val_logits, tile_logits_by_variant, log_space=True):
    T_ = fit_temperature(val_logits, Y[VA])
    for v in VARIANTS:
        z = tile_logits_by_variant[v]
        if log_space:
            lg = aggregate(VAR_ITEMS[v], z, 'mean')
        else:                                   # z are probabilities: average, then log
            lg = np.log(aggregate(VAR_ITEMS[v], z, 'mean') + 1e-12)
        PRED[method][v][seed] = softmax_np(lg / T_)


comparators = {}
for name in CFG['backbones']:
    clf = fit_logreg(FE[name]['X'][TR], Y[TR], FE[name]['X'][VA], Y[VA])
    comparators[f'LinProbe {name} (frozen)'] = (lambda clf=clf, name=name: (np.log(full_proba(clf, FE[name]['X'][VA]) + 1e-12),
                                                                            {v: full_proba(clf, FE[name]['V'][v]) for v in VARIANTS}))
clf = fit_logreg(Xc[TR], Y[TR], Xc[VA], Y[VA])
comparators['LogReg concat (frozen)'] = lambda clf=clf: (np.log(full_proba(clf, Xc[VA]) + 1e-12), {v: full_proba(clf, Vc[v]) for v in VARIANTS})
pca = PCA(min(CFG['svm_pca'], len(TR) - 1), random_state=0).fit(Xc[TR])
svm = SVC(C=CFG['svm_C'], gamma='scale', probability=True, class_weight='balanced', random_state=0).fit(pca.transform(Xc[TR]), Y[TR])
comparators['RBF-SVM concat (frozen)'] = lambda: (np.log(full_proba(svm, pca.transform(Xc[VA])) + 1e-12), {v: full_proba(svm, pca.transform(Vc[v])) for v in VARIANTS})
knn = KNeighborsClassifier(CFG['knn_k'], metric='cosine', weights='distance').fit(Xc[TR], Y[TR])
comparators['k-NN concat (frozen)'] = lambda: (np.log(full_proba(knn, Xc[VA]) + 1e-12), {v: full_proba(knn, Vc[v]) for v in VARIANTS})
for m, f in comparators.items():
    lv, probs = f()
    register(m, 0, lv, probs, log_space=False)
for s in CFG['seeds']:
    mlp = train_mlp(Xc[TR], Y[TR], Xc[VA], Y[VA], s)
    register('MLP head concat (frozen)', s, mlp(Xc[VA]), {v: mlp(Vc[v]) for v in VARIANTS}, log_space=True)
print('frozen comparators done')

# ---- proposed model
OURS = 'LoRA Phikon-v2 + stain aug (ours)'
for s in CFG['seeds']:
    t0 = time.time()
    model = train_peft(s)
    lv = peft_logits(model, PATHS[VA])
    register(OURS, s, lv, {v: peft_logits(model, [p for _, p in VAR_ITEMS[v]]) for v in VARIANTS}, log_space=True)
    del model
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    print(f'  OURS seed {s} done in {(time.time()-t0)/60:.1f} min')

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

[dinov2_vitb14] features ready


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

...

    seed 0 epoch 1/6: val bal_acc 0.7634 nll 0.4065 (228s)
    seed 0 epoch 2/6: val bal_acc 0.7393 nll 0.3700 (227s)
    seed 0 epoch 3/6: val bal_acc 0.7669 nll 0.3869 (227s)
    seed 0 epoch 4/6: val bal_acc 0.7613 nll 0.3653 (227s)
    seed 0 epoch 5/6: val bal_acc 0.7600 nll 0.3695 (227s)
    seed 0 epoch 6/6: val bal_acc 0.7469 nll 0.3494 (227s)
  OURS seed 0 done in 33.0 min


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    seed 1 epoch 1/6: val bal_acc 0.7747 nll 0.4138 (227s)
    seed 1 epoch 2/6: val bal_acc 0.7565 nll 0.3168 (227s)
    seed 1 epoch 3/6: val bal_acc 0.7870 nll 0.4064 (227s)
    seed 1 epoch 4/6: val bal_acc 0.7574 nll 0.3854 (227s)
    seed 1 epoch 5/6: val bal_acc 0.7527 nll 0.3235 (227s)
    seed 1 epoch 6/6: val bal_acc 0.7645 nll 0.3513 (227s)
  OURS seed 1 done in 32.9 min


Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

    seed 2 epoch 1/6: val bal_acc 0.7410 nll 0.3309 (227s)
    seed 2 epoch 2/6: val bal_acc 0.7714 nll 0.3166 (227s)
    seed 2 epoch 3/6: val bal_acc 0.7707 nll 0.3512 (227s)
    seed 2 epoch 4/6: val bal_acc 0.7424 nll 0.3303 (227s)
    seed 2 epoch 5/6: val bal_acc 0.7500 nll 0.3102 (227s)
  OURS seed 2 done in 29.1 min


## 8. H3 — EM prior-shift correction (Saerens et al. 2002), then metrics for all 8 conditions

In [8]:
def em_prior(P, prior_tr, iters=200, tol=1e-6):
    pi = prior_tr.copy()
    for _ in range(iters):
        Q = P * (pi / prior_tr); Q /= Q.sum(1, keepdims=True)
        new = Q.mean(0)
        if np.abs(new - pi).max() < tol:
            pi = new; break
        pi = new
    Q = P * (pi / prior_tr); Q /= Q.sum(1, keepdims=True)
    return Q, pi


def ece(P, y, n_bins=15):
    conf, pred, e = P.max(1), P.argmax(1), 0.0
    for lo, hi in zip(np.linspace(0, 1, n_bins + 1)[:-1], np.linspace(0, 1, n_bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e


yB, clsB = BACH.y.values, BACH.cls.values
FINAL = defaultdict(lambda: defaultdict(dict))     # FINAL[method][condition][seed] -> probs
rows, est_prior = [], []
for m in PRED:
    for v in VARIANTS:
        for s, P in PRED[m][v].items():
            for post in ['none', 'EM prior']:
                if post == 'none':
                    Q = P
                else:
                    Q, pi = em_prior(P, PRIOR_TR)
                    est_prior.append(dict(method=m, variant=v, seed=s, est_malignant_prior=pi[1]))
                cond = f'{v} | {post}'
                FINAL[m][cond][s] = Q
                pred = Q.argmax(1)
                compat = np.isin(clsB, ['Benign', 'Invasive'])
                rows.append(dict(method=m, input=v, post=post, condition=cond, seed=s,
                                 bal_acc=balanced_accuracy_score(yB, pred), auc=roc_auc_score(yB, Q[:, 1]), macro_f1=f1_score(yB, pred, average='macro'),
                                 sensitivity=(pred[yB == 1] == 1).mean(), specificity=(pred[yB == 0] == 0).mean(), ece=ece(Q, yB),
                                 compat_bal_acc=balanced_accuracy_score(yB[compat], pred[compat]),
                                 **{f'malig_rate_{c}': (pred[clsB == c] == 1).mean() for c in ['Normal', 'Benign', 'InSitu', 'Invasive']}))
RB = pd.DataFrame(rows)
RB.to_csv(RES_DIR / 'bach_all_rows.csv', index=False)
pd.DataFrame(est_prior).to_csv(RES_DIR / 'bach_em_estimated_prior.csv', index=False)
print('estimated BACH malignant prior (EM, mean over methods/seeds):', round(pd.DataFrame(est_prior).est_malignant_prior.mean(), 3), '(true: 0.5)')

estimated BACH malignant prior (EM, mean over methods/seeds): 0.585 (true: 0.5)


## 9. Tables and significance

In [9]:
pd.set_option('display.width', 250)


def save_table(t, name, caption):
    t.to_csv(TAB_DIR / f'{name}.csv')
    try:
        t.to_latex(TAB_DIR / f'{name}.tex', caption=caption, label=f'tab:{name}', escape=True)
    except Exception:
        pass
    print(f'\n=== {name}: {caption} ===\n{t.to_string()}')


COND_ORDER = [f'{v} | {p}' for v in VARIANTS for p in ['none', 'EM prior']]
pivot = RB.groupby(['method', 'condition']).bal_acc.agg(['mean', 'std']).reset_index()
pivot['cell'] = [f'{m*100:.1f} ± {0 if np.isnan(s) else s*100:.1f}' for m, s in zip(pivot['mean'], pivot['std'])]
TB1 = pivot.pivot(index='method', columns='condition', values='cell')[COND_ORDER]
save_table(TB1, 'TB1_bach_bal_acc_by_condition', 'BACH balanced accuracy (%) for every method × condition (mean ± std over seeds).')
TB2 = (RB.groupby(['method', 'condition'])[['bal_acc', 'auc', 'sensitivity', 'specificity', 'macro_f1', 'ece', 'compat_bal_acc']].mean() * np.array([100, 100, 100, 100, 100, 1, 100])).round(2)
save_table(TB2, 'TB2_bach_all_metrics', 'All BACH metrics per method and condition (mean over seeds; ECE unscaled).')
TB3 = (RB[RB.method == OURS].groupby('condition')[[f'malig_rate_{c}' for c in ['Normal', 'Benign', 'InSitu', 'Invasive']]].mean() * 100).round(1).reindex(COND_ORDER)
save_table(TB3, 'TB3_ours_malignant_rate_per_bach_class', 'Proposed model: % of each original BACH class predicted malignant (H4: label-space mismatch).')


def boot_diff(y, a, b, n_boot, seed=0):
    rng_ = np.random.default_rng(seed)
    f = lambda ii: balanced_accuracy_score(y[ii], a[ii]) - balanced_accuracy_score(y[ii], b[ii])
    d = f(np.arange(len(y)))
    diffs = np.array([f(rng_.integers(0, len(y), len(y))) for _ in range(n_boot)])
    return d, np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)


BASE, FULL = 'resize (previous) | none', 'tiles + stain-norm | EM prior'
sig = []
for m in FINAL:
    for cond in COND_ORDER:
        if cond == BASE:
            continue
        res = [boot_diff(yB, FINAL[m][cond][s].argmax(1), FINAL[m][BASE][s].argmax(1), CFG['n_boot'], s) for s in FINAL[m][cond]]
        n_up, n_dn = sum(r[1] > 0 for r in res), sum(r[2] < 0 for r in res)
        sig.append(dict(method=m, condition=cond, delta_pp=np.mean([r[0] for r in res]) * 100, ci_lo_seed0=res[0][1] * 100, ci_hi_seed0=res[0][2] * 100,
                        seeds_improved=f'{n_up}/{len(res)}', seeds_worse=f'{n_dn}/{len(res)}',
                        verdict='improves (all seeds)' if n_up == len(res) else 'worse (all seeds)' if n_dn == len(res) else
                        'mixed' if (n_up and n_dn) else 'improves (some seeds)' if n_up else 'worse (some seeds)' if n_dn else 'no significant change'))
SIG = pd.DataFrame(sig)
save_table(SIG[SIG.method == OURS].set_index('condition').drop(columns='method').round(3), 'TB4_ours_condition_vs_previous',
           'Proposed model: each condition vs the previous pipeline (resize | none); paired image bootstrap.')
save_table(SIG.pivot(index='method', columns='condition', values='verdict')[[c for c in COND_ORDER if c != BASE]], 'TB5_all_methods_effect',
           'Effect of each condition vs the previous pipeline, for every method.')
# best condition comparison: ours vs comparators under the FULL pipeline
cmp_rows = []
for m in FINAL:
    if m == OURS:
        continue
    res = []
    for s in FINAL[OURS][FULL]:
        Pc = FINAL[m][FULL].get(s, FINAL[m][FULL][min(FINAL[m][FULL])])
        res.append(boot_diff(yB, FINAL[OURS][FULL][s].argmax(1), Pc.argmax(1), CFG['n_boot'], s))
    n_up, n_dn = sum(r[1] > 0 for r in res), sum(r[2] < 0 for r in res)
    cmp_rows.append(dict(comparator=m, delta_pp=np.mean([r[0] for r in res]) * 100, ci_lo_seed0=res[0][1] * 100, ci_hi_seed0=res[0][2] * 100,
                         ours_better_seeds=f'{n_up}/{len(res)}', ours_worse_seeds=f'{n_dn}/{len(res)}'))
save_table(pd.DataFrame(cmp_rows).set_index('comparator').round(3), 'TB6_ours_vs_comparators_full_pipeline',
           f'Under the full pipeline ({FULL}): proposed model vs each comparator.')


=== TB1_bach_bal_acc_by_condition: BACH balanced accuracy (%) for every method × condition (mean ± std over seeds). ===
condition                         resize (previous) | none resize (previous) | EM prior tiles | none tiles | EM prior stain-norm | none stain-norm | EM prior tiles + stain-norm | none tiles + stain-norm | EM prior
method                                                                                                                                                                                                               
LinProbe dinov2_vitb14 (frozen)                 57.8 ± 0.0                   60.2 ± 0.0   61.0 ± 0.0       52.5 ± 0.0        59.0 ± 0.0            50.2 ± 0.0                52.0 ± 0.0                    50.0 ± 0.0
LinProbe phikon_v2 (frozen)                     50.0 ± 0.0                   50.0 ± 0.0   65.5 ± 0.0       68.8 ± 0.0        50.2 ± 0.0            50.0 ± 0.0                63.5 ± 0.0                    68.0 ± 0.0
LoRA Phikon-v2 + stain 

## 10. Pre-registered decision

In [10]:
r = SIG[(SIG.method == OURS) & (SIG.condition == FULL)].iloc[0]
print('PRIMARY HYPOTHESIS: full pipeline improves the proposed model on BACH (all seeds)')
print(f"  -> {'SUPPORTED' if r.verdict == 'improves (all seeds)' else 'NOT SUPPORTED'}: Δ = {r.delta_pp:+.2f} pp, seed-0 CI [{r.ci_lo_seed0:.2f}, {r.ci_hi_seed0:.2f}], improved in {r.seeds_improved} seeds")
for cond, h in [('tiles | none', 'H1 scale-matched tiling'), ('stain-norm | none', 'H2 stain normalisation'), ('resize (previous) | EM prior', 'H3 EM prior correction')]:
    q = SIG[(SIG.method == OURS) & (SIG.condition == cond)].iloc[0]
    print(f"  {h:28s}: {q.verdict:24s} Δ = {q.delta_pp:+.2f} pp (seeds improved {q.seeds_improved}, worse {q.seeds_worse})")
print(f"  H1 scale estimate valid: {SCALE_OK} (r = {SCALE_R:.3f})")
t3 = TB3.loc[BASE]
print(f"  H4 label-space: previous pipeline predicts malignant for {t3['malig_rate_Normal']:.0f}% of Normal and {t3['malig_rate_Benign']:.0f}% of Benign BACH images")
pd.DataFrame([dict(hypothesis='primary: full pipeline improves ours (all seeds)', supported=r.verdict == 'improves (all seeds)',
                   delta_pp=r.delta_pp, seeds=r.seeds_improved)]).to_csv(TAB_DIR / 'bach_preregistered_decision.csv', index=False)

PRIMARY HYPOTHESIS: full pipeline improves the proposed model on BACH (all seeds)
  -> NOT SUPPORTED: Δ = -8.00 pp, seed-0 CI [-19.77, -12.40], improved in 0/3 seeds
  H1 scale-matched tiling     : improves (all seeds)     Δ = +11.33 pp (seeds improved 3/3, worse 0/3)
  H2 stain normalisation      : worse (some seeds)       Δ = -0.67 pp (seeds improved 0/3, worse 1/3)
  H3 EM prior correction      : worse (all seeds)        Δ = -10.50 pp (seeds improved 0/3, worse 3/3)
  H1 scale estimate valid: False (r = 1.000)
  H4 label-space: previous pipeline predicts malignant for 73% of Normal and 70% of Benign BACH images


## 11. Figures

In [11]:
OKABE = ['#0072B2', '#E69F00', '#009E73', '#D55E00', '#CC79A7', '#56B4E9', '#F0E442', '#000000']
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})


def save(fig, name):
    fig.tight_layout(); fig.savefig(FIG_DIR / f'{name}.png', dpi=300, bbox_inches='tight'); fig.savefig(FIG_DIR / f'{name}.pdf', bbox_inches='tight'); plt.close(fig); print('saved', name)


short = lambda m: m.replace(' (frozen)', '*').replace(OURS, 'Ours (LoRA Phikon-v2)')
H = RB.groupby(['method', 'condition']).bal_acc.mean().unstack()[COND_ORDER] * 100
H = H.loc[[OURS] + [m for m in H.index if m != OURS]]
fig, ax = plt.subplots(figsize=(11, 0.5 * len(H) + 2))
im = ax.imshow(H.values, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(COND_ORDER))); ax.set_xticklabels([c.replace(' | ', '\n') for c in COND_ORDER], fontsize=7.5)
ax.set_yticks(range(len(H))); ax.set_yticklabels([short(m) for m in H.index])
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        ax.text(j, i, f'{H.values[i, j]:.1f}', ha='center', va='center', color='w' if H.values[i, j] < H.values.mean() else 'k', fontsize=8)
fig.colorbar(im, label='BACH balanced accuracy (%)'); ax.set_title('Trained on BreakHis, tested on BACH: every method × input × post-hoc condition')
save(fig, 'FB2_condition_heatmap')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, cond in enumerate([BASE, 'tiles | none', 'stain-norm | none', FULL]):
    P = FINAL[OURS][cond][min(FINAL[OURS][cond])]
    fpr, tpr, _ = roc_curve(yB, P[:, 1]); axes[0].plot(fpr, tpr, color=OKABE[i], lw=2 if cond == FULL else 1.2, label=f'{cond} (AUC {roc_auc_score(yB, P[:, 1]):.3f})')
axes[0].plot([0, 1], [0, 1], '--', color='grey'); axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].legend(frameon=False, fontsize=7); axes[0].set_title('Ours on BACH — ROC by condition')
x = np.arange(4); w = 0.2
for i, cond in enumerate([BASE, 'tiles | none', 'stain-norm | none', FULL]):
    axes[1].bar(x + (i - 1.5) * w, TB3.loc[cond].values, w, color=OKABE[i], label=cond)
axes[1].set_xticks(x); axes[1].set_xticklabels(['Normal', 'Benign', 'InSitu', 'Invasive']); axes[1].set_ylabel('% predicted malignant')
axes[1].axhline(50, ls=':', color='grey'); axes[1].legend(frameon=False, fontsize=6.5); axes[1].set_title('Ours: malignant rate per original BACH class (H4)')
save(fig, 'FB3_ours_roc_and_class_rates')

fig, ax = plt.subplots(figsize=(5.2, 3.6))
for i, mag in enumerate(MAGS):
    ax.scatter(np.full((SC.mag == mag).sum(), i), SC[SC.mag == mag].d, s=10, color=OKABE[0], alpha=0.6)
ax.scatter(np.full((SC.source == 'BACH').sum(), 4), SC[SC.source == 'BACH'].d, s=10, color=OKABE[3], alpha=0.6)
ax.plot(range(4), bh, '-o', color=OKABE[0]); ax.axhline(med['BACH'], ls='--', color=OKABE[3])
ax.set_xticks(range(5)); ax.set_xticklabels([f'BH {m}X' for m in MAGS] + ['BACH']); ax.set_ylabel('median nucleus diameter (px)')
ax.set_title(f'H1 scale estimate: r = {SCALE_R:.2f} ({"valid" if SCALE_OK else "fallback"})'); save(fig, 'FB4_scale_estimate')

saved FB2_condition_heatmap
saved FB3_ours_roc_and_class_rates
saved FB4_scale_estimate


## 12. Export

In [12]:
with open(RES_DIR / 'bach_bundle.pkl', 'wb') as fh:
    pickle.dump(dict(rows=RB, final={m: {c: dict(d) for c, d in v.items()} for m, v in FINAL.items()}, bach=BACH, scale=dict(r=SCALE_R, ok=SCALE_OK),
                     prior_tr=PRIOR_TR, cfg=CFG), fh)
for d in ('results', 'tables', 'figures'):
    shutil.make_archive(str(WORK / f'bach_{d}'), 'zip', root_dir=WORK, base_dir=d)
print('Done. Output tab: bach_tables.zip, bach_figures.zip, bach_results.zip')

Done. Output tab: bach_tables.zip, bach_figures.zip, bach_results.zip
